In [ ]:
"""
Eagles 2025 Google News -> article scraper (Selenium)

Goal:
- For each of 17 Eagles 2025 regular-season games:
  - Collect 20 *NON-PAYWALLED* articles
  - Each output row: source, author, title, text_body (+ helpful fields)
- Paywalled articles are skipped and DO NOT count toward the 20.
- The script paginates Google News results and “oversamples” links until it reaches 20
  non-paywalled rows or runs out of results.

Dependencies:
  pip install selenium beautifulsoup4

Notes:
- Google can show consent/CAPTCHA (“unusual traffic”). Run HEADLESS=False first.
- “Author” often missing; extracted from meta tags when present.
- Text-body extraction is heuristic; paywalls + JS-heavy sites can reduce quality.

IMPORTANT:
- The GAMES_2025 list below should be verified against a canonical schedule source you trust.
  (If you paste your exact schedule source link, you should update the list accordingly.)
"""

from __future__ import annotations

import csv
import random
import re
import time
from typing import Dict, List, Optional, Tuple
from urllib.parse import quote_plus, urlparse

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
    NoSuchElementException,
)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ----------------------------
# CONFIG
# ----------------------------

TARGET_PER_GAME = 20
OUT_CSV = "../google_news_outputs/eagles_2025_articles_non_paywalled.csv"
HEADLESS = False

PAGELOAD_TIMEOUT_S = 25
WAIT_S = 12

# Throttling
SLEEP_BETWEEN_ARTICLES = (2.5, 6.0)   # between opening articles
SLEEP_BETWEEN_PAGES = (2.0, 5.0)      # between paginating search results
SLEEP_BETWEEN_GAMES = (10.0, 25.0)    # between games
BACKOFF_BASE = 3.0

# Google search result count hint (Google may ignore)
GOOGLE_NEWS_NUM_PER_PAGE = 20

# Safety: don't paginate forever per game
MAX_RESULTS_PAGES_PER_GAME = 10       # cap Google pagination
MAX_ARTICLE_VISITS_PER_GAME = 120     # cap visits in case of many paywalls/blocks


# ----------------------------
# EAGLES 2025 REGULAR SEASON (17 GAMES)
# Verify/replace as needed
# ----------------------------

GAMES_2025: List[Dict[str, str]] = [
    {"week": "1",  "date": "2025-09-04", "opponent": "Dallas Cowboys",         "home_away": "home"},
    {"week": "2",  "date": "2025-09-14", "opponent": "Kansas City Chiefs",    "home_away": "away"},
    {"week": "3",  "date": "2025-09-21", "opponent": "Los Angeles Rams",      "home_away": "home"},
    {"week": "4",  "date": "2025-09-28", "opponent": "Tampa Bay Buccaneers",  "home_away": "away"},
    {"week": "5",  "date": "2025-10-05", "opponent": "Denver Broncos",        "home_away": "home"},
    {"week": "6",  "date": "2025-10-09", "opponent": "New York Giants",       "home_away": "away"},
    {"week": "7",  "date": "2025-10-19", "opponent": "Minnesota Vikings",     "home_away": "away"},
    {"week": "8",  "date": "2025-10-26", "opponent": "New York Giants",       "home_away": "home"},
    {"week": "10", "date": "2025-11-10", "opponent": "Green Bay Packers",     "home_away": "away"},
    {"week": "11", "date": "2025-11-16", "opponent": "Detroit Lions",         "home_away": "home"},
    {"week": "12", "date": "2025-11-23", "opponent": "Dallas Cowboys",        "home_away": "away"},
    {"week": "13", "date": "2025-11-28", "opponent": "Chicago Bears",         "home_away": "home"},
    {"week": "14", "date": "2025-12-08", "opponent": "Los Angeles Chargers",  "home_away": "away"},
    {"week": "15", "date": "2025-12-14", "opponent": "Las Vegas Raiders",     "home_away": "home"},
    {"week": "16", "date": "2025-12-20", "opponent": "Washington Commanders", "home_away": "away"},
    {"week": "17", "date": "2025-12-28", "opponent": "Buffalo Bills",         "home_away": "away"},
    {"week": "18", "date": "2026-01-04", "opponent": "Washington Commanders", "home_away": "home"},
]


# ----------------------------
# PAYWALL DETECTION
# ----------------------------

PAYWALL_KEYWORDS = [
    "subscribe to continue",
    "subscribe now",
    "subscription required",
    "subscriber-only",
    "sign in to continue",
    "register to continue",
    "create an account",
    "already a subscriber",
    "to keep reading",
    "unlimited access",
    "unlock this article",
    "this content is available to subscribers",
    "this article is for subscribers",
    "become a subscriber",
    "membership required",
    "you've reached your limit",
    "free articles remaining",
    "metered",
    "turn off your ad blocker",
    "tinypass",
    "piano",
    "paywall",
    "zephr",
    "subscribe with google",
]

PAYWALL_CSS_SELECTORS = [
    '[id*="paywall" i]',
    '[class*="paywall" i]',
    '[id*="subscribe" i]',
    '[class*="subscribe" i]',
    '[id*="subscription" i]',
    '[class*="subscription" i]',
    '[class*="meter" i]',
    '[id*="meter" i]',
    '[class*="tp-" i]',
    '[id*="tp-" i]',
    '[class*="piano" i]',
    '[id*="piano" i]',
    '[class*="zephr" i]',
    '[id*="zephr" i]',
    '[class*="regwall" i]',
    '[id*="regwall" i]',
    '[class*="overlay" i]',
    '[class*="modal" i]',
]

VENDOR_MARKERS = [
    "tinypass",
    "piano.io",
    "sandbox.piano",
    "zephr",
    "arcxp",
    "gateway",
]


# ----------------------------
# UTIL
# ----------------------------

def jitter_sleep(a_b: Tuple[float, float]) -> None:
    time.sleep(random.uniform(*a_b))

def safe_get(driver: webdriver.Chrome, url: str, retries: int = 3) -> None:
    for attempt in range(1, retries + 1):
        try:
            driver.set_page_load_timeout(PAGELOAD_TIMEOUT_S)
            driver.get(url)
            return
        except (TimeoutException, WebDriverException):
            if attempt == retries:
                raise
            time.sleep(BACKOFF_BASE * attempt + random.uniform(0.0, 2.0))

def likely_google_consent_or_block(html: str) -> bool:
    s = (html or "").lower()
    return (
        ("before you continue to google" in s)
        or ("consent.google.com" in s)
        or ("unusual traffic" in s)
        or ("our systems have detected unusual traffic" in s)
        or ("sorry" in s and "robots" in s)
    )

def build_google_news_search_url(query: str, num: int = 20) -> str:
    q = quote_plus(query)
    return f"https://www.google.com/search?q={q}&tbm=nws&num={num}&hl=en&gl=us"

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).strip()

def canonicalize_url(url: str) -> str:
    try:
        u = urlparse(url)
        return f"{u.scheme}://{u.netloc}{u.path}"
    except Exception:
        return url


# ----------------------------
# PAYWALL HELPERS
# ----------------------------

def looks_like_paywall_html(html: str) -> Tuple[bool, str]:
    low = (html or "").lower()
    hits = [kw for kw in PAYWALL_KEYWORDS if kw in low]
    if hits:
        return (True, f"keyword:{hits[0]}")
    for m in VENDOR_MARKERS:
        if m in low:
            return (True, f"vendor_marker:{m}")
    return (False, "")

def looks_like_paywall_dom(driver: webdriver.Chrome) -> Tuple[bool, str]:
    try:
        try:
            body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
        except Exception:
            body_text = ""

        for kw in PAYWALL_KEYWORDS:
            if kw in body_text:
                return (True, f"body_text:{kw}")

        for sel in PAYWALL_CSS_SELECTORS:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if not els:
                continue
            for el in els:
                try:
                    if el.is_displayed():
                        # don't over-trigger on generic modal/overlay unless it contains key terms
                        if ("modal" in sel or "overlay" in sel):
                            t = (el.text or "").lower()
                            if any(k in t for k in ["subscribe", "subscriber", "sign in", "register", "limit"]):
                                return (True, f"selector:{sel}")
                            continue
                        return (True, f"selector:{sel}")
                except Exception:
                    continue

        return (False, "")
    except Exception:
        return (False, "")

def is_probably_paywalled(driver: webdriver.Chrome, html: str) -> Tuple[bool, str]:
    pw1, r1 = looks_like_paywall_html(html)
    if pw1:
        return (True, r1)
    pw2, r2 = looks_like_paywall_dom(driver)
    if pw2:
        return (True, r2)
    return (False, "")


# ----------------------------
# EXTRACTION (author, title, body)
# ----------------------------

def extract_author_from_meta(soup: BeautifulSoup) -> Optional[str]:
    selectors = [
        'meta[name="author"]',
        'meta[property="article:author"]',
        'meta[name="parsely-author"]',
    ]
    for sel in selectors:
        tag = soup.select_one(sel)
        if tag:
            c = clean_text(tag.get("content", ""))
            if c:
                return re.sub(r"^\s*by\s+", "", c, flags=re.I).strip()
    return None

def extract_title_from_article(soup: BeautifulSoup) -> Optional[str]:
    og = soup.select_one('meta[property="og:title"]')
    if og and clean_text(og.get("content", "")):
        return clean_text(og.get("content", ""))
    if soup.title and soup.title.string:
        return clean_text(soup.title.string)
    h1 = soup.find("h1")
    if h1:
        return clean_text(h1.get_text(" ", strip=True))
    return None

def extract_main_text_from_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg", "canvas", "header", "footer", "aside"]):
        tag.decompose()

    article = soup.find("article")
    if article:
        txt = clean_text(article.get_text(" ", strip=True))
        if len(txt) >= 400:
            return txt

    main = soup.find("main")
    if main:
        txt = clean_text(main.get_text(" ", strip=True))
        if len(txt) >= 400:
            return txt

    best_txt, best_score = "", 0
    for node in soup.find_all(["div", "section"]):
        txt = clean_text(node.get_text(" ", strip=True))
        if len(txt) < 400:
            continue

        punct = len(re.findall(r"[.!?]", txt))
        score = len(txt) + 50 * min(punct, 30)

        class_id = " ".join([
            " ".join(node.get("class", []) or []),
            node.get("id", "") or ""
        ]).lower()
        if any(k in class_id for k in ["nav", "menu", "footer", "header", "sidebar", "subscribe", "promo"]):
            score *= 0.6

        if score > best_score:
            best_score = score
            best_txt = txt

    return best_txt


# ----------------------------
# GOOGLE NEWS RESULTS PARSING
# ----------------------------

def collect_result_cards(driver: webdriver.Chrome) -> List[BeautifulSoup]:
    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.select("div.SoaBEf")
    if cards:
        return cards

    anchors = soup.select("a.WlydOe")
    out = []
    for a in anchors:
        parent = a.find_parent("g-card") or a.find_parent("div")
        if parent:
            out.append(parent)
    return out

def parse_card(card: BeautifulSoup) -> Dict[str, Optional[str]]:
    a = card.select_one("a.WlydOe")
    title = clean_text(a.get_text(" ", strip=True)) if a else None
    link = a.get("href") if a else None

    if link and link.startswith("/"):
        link = "https://www.google.com" + link

    source = None
    src = card.select_one(".CEMjEf, .MgUUmf span, span.xQ82C")
    if src:
        source = clean_text(src.get_text(" ", strip=True))

    return {"title": title, "link": link, "source": source}


# ----------------------------
# DRIVER
# ----------------------------

def start_driver() -> webdriver.Chrome:
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")

    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")

    # best-effort anti-automation flags (not magic)
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)

    opts.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=opts)
    try:
        driver.execute_cdp_cmd(
            "Page.addScriptToEvaluateOnNewDocument",
            {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined});"}
        )
    except Exception:
        pass

    return driver


# ----------------------------
# SCRAPE FLOW
# ----------------------------

def build_game_query(game: Dict[str, str]) -> str:
    week = game["week"]
    date = game["date"]
    opp = game["opponent"]
    ha = "at" if game["home_away"] == "away" else "vs"
    return f'Philadelphia Eagles {ha} "{opp}" {date} week {week} recap analysis'

def open_in_new_tab(driver: webdriver.Chrome, url: str) -> None:
    driver.execute_script("window.open(arguments[0], '_blank');", url)
    driver.switch_to.window(driver.window_handles[-1])

def close_current_tab_and_return(driver: webdriver.Chrome) -> None:
    driver.close()
    driver.switch_to.window(driver.window_handles[0])

def click_next_results_page(driver: webdriver.Chrome) -> bool:
    try:
        nxt = driver.find_element(By.CSS_SELECTOR, "a#pnnext")
        jitter_sleep((1.0, 2.5))
        nxt.click()
        return True
    except NoSuchElementException:
        return False
    except Exception:
        return False

def ensure_results_loaded(driver: webdriver.Chrome) -> None:
    WebDriverWait(driver, WAIT_S).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.WlydOe, div.SoaBEf"))
    )

def scrape_non_paywalled_for_game(driver: webdriver.Chrome, game: Dict[str, str], target: int) -> List[Dict[str, str]]:
    """
    Visits articles until it accumulates `target` non-paywalled rows,
    paginating Google News results as needed.
    """
    query = build_game_query(game)
    search_url = build_google_news_search_url(query, num=GOOGLE_NEWS_NUM_PER_PAGE)

    # Open search
    safe_get(driver, search_url)

    if likely_google_consent_or_block(driver.page_source):
        print("[!] Google consent/block page detected.")
        print("    Complete consent/CAPTCHA in the opened browser window, then re-run this game.")
        time.sleep(10)

    ensure_results_loaded(driver)

    seen_links = set()            # canonicalized
    rows: List[Dict[str, str]] = []
    pages_used = 0
    visits = 0

    while len(rows) < target:
        if pages_used >= MAX_RESULTS_PAGES_PER_GAME:
            break
        if visits >= MAX_ARTICLE_VISITS_PER_GAME:
            break

        # parse current page cards
        cards = collect_result_cards(driver)
        new_refs: List[Dict[str, str]] = []
        for c in cards:
            d = parse_card(c)
            if not d.get("link") or not d.get("title"):
                continue
            can = canonicalize_url(d["link"])
            if can in seen_links:
                continue
            seen_links.add(can)
            new_refs.append({
                "source": d.get("source") or "",
                "title_hint": d.get("title") or "",
                "link": d["link"],
            })

        # visit each new ref until target met
        for ref in new_refs:
            if len(rows) >= target:
                break
            if visits >= MAX_ARTICLE_VISITS_PER_GAME:
                break

            visits += 1
            jitter_sleep(SLEEP_BETWEEN_ARTICLES)

            link = ref["link"]
            try:
                open_in_new_tab(driver, link)
                time.sleep(random.uniform(3.0, 7.0))

                html = driver.page_source

                paywalled, _reason = is_probably_paywalled(driver, html)
                if paywalled:
                    # skip — does not count toward the 20
                    continue

                soup = BeautifulSoup(html, "html.parser")
                author = extract_author_from_meta(soup) or ""
                title = extract_title_from_article(soup) or ref.get("title_hint", "")
                body = extract_main_text_from_html(html) or ""

                # Soft paywall / junk guard:
                # If body is extremely short AND page has subscribe-ish terms, treat as paywalled/junk skip.
                low = html.lower()
                if len(body) < 250 and any(k in low for k in ["subscribe", "subscriber", "sign in", "register", "limit"]):
                    continue

                # Basic minimum content (helps avoid nav pages / cookie-only pages)
                if len(body) < 250:
                    continue

                rows.append({
                    "week": game["week"],
                    "game_date": game["date"],
                    "opponent": game["opponent"],
                    "home_away": game["home_away"],
                    "query": query,
                    "source": ref.get("source", ""),
                    "author": author,
                    "title": title,
                    "text_body": body,
                    "url": link,
                })

            except Exception as e:
                print(f"[!] Article error week {game['week']}: {e}")
            finally:
                # close article tab
                if len(driver.window_handles) > 1:
                    try:
                        close_current_tab_and_return(driver)
                    except Exception:
                        driver.switch_to.window(driver.window_handles[0])

        # If still short, paginate
        if len(rows) < target:
            pages_used += 1
            jitter_sleep(SLEEP_BETWEEN_PAGES)

            moved = click_next_results_page(driver)
            if not moved:
                break

            # allow load
            time.sleep(random.uniform(2.0, 4.0))
            if likely_google_consent_or_block(driver.page_source):
                print("[!] Hit Google consent/block mid-pagination; stopping this game early.")
                break
            try:
                ensure_results_loaded(driver)
            except TimeoutException:
                break

    return rows

def run_all_games(games: List[Dict[str, str]], target_per_game: int = 20, out_csv: str = OUT_CSV) -> List[Dict[str, str]]:
    if len(games) != 17:
        print(f"[!] Provided {len(games)} games; expected 17.")

    driver = start_driver()
    all_rows: List[Dict[str, str]] = []

    fieldnames = [
        "week", "game_date", "opponent", "home_away", "query",
        "source", "author", "title", "text_body", "url",
    ]

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        try:
            for gi, game in enumerate(games, 1):
                print(
                    f"\n=== Game {gi}/{len(games)} | Week {game['week']} "
                    f"({game['date']}) {game['home_away']} vs {game['opponent']} ==="
                )

                try:
                    rows = scrape_non_paywalled_for_game(driver, game, target=target_per_game)
                except TimeoutException:
                    print("[!] Timeout on results page; backing off and continuing.")
                    time.sleep(BACKOFF_BASE + random.uniform(0.0, 4.0))
                    rows = []
                except WebDriverException as e:
                    print(f"[!] WebDriver error: {e}. Backing off and continuing.")
                    time.sleep(BACKOFF_BASE + random.uniform(0.0, 6.0))
                    rows = []

                print(f"Collected {len(rows)} non-paywalled articles for week {game['week']}.")

                for r in rows:
                    writer.writerow(r)
                f.flush()

                all_rows.extend(rows)

                jitter_sleep(SLEEP_BETWEEN_GAMES)

        finally:
            try:
                driver.quit()
            except Exception:
                pass

    print(f"\nDone. Wrote {len(all_rows)} rows to {out_csv}.")
    return all_rows


# ----------------------------
# RUN
# ----------------------------
rows = run_all_games(GAMES_2025, target_per_game=TARGET_PER_GAME, out_csv=OUT_CSV)



=== Game 1/17 | Week 1 (2025-09-04) home vs Dallas Cowboys ===
[!] Error article 5/14 week 1: Message: timeout: Timed out receiving message from renderer: 25.000
  (Session info: chrome=145.0.7632.76)
Stacktrace:
0   chromedriver                        0x00000001048cfd64 cxxbridge1$str$ptr + 3127864
1   chromedriver                        0x00000001048c8154 cxxbridge1$str$ptr + 3096104
2   chromedriver                        0x00000001043a59f4 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75356
3   chromedriver                        0x000000010438ff00 cxxbridge1$string$len + 4080
4   chromedriver                        0x000000010438fc70 cxxbridge1$string$len + 3424
5   chromedriver                        0x000000010438d9ec chromedriver + 219628
6   chromedriver                        0x000000010438e5e0 chromedriver + 222688
7   chromedriver                        0x000000010439bc38 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 34976
8   ch

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
0   chromedriver                        0x00000001048cfd64 cxxbridge1$str$ptr + 3127864
1   chromedriver                        0x00000001048c8154 cxxbridge1$str$ptr + 3096104
2   chromedriver                        0x00000001043a5840 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74920
3   chromedriver                        0x00000001043e1628 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 320144
4   chromedriver                        0x000000010440a214 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 487036
5   chromedriver                        0x00000001044096ec _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 484180
6   chromedriver                        0x0000000104373564 chromedriver + 11